In [2]:
import os
import numpy as np
from  rank_bm25 import BM25Okapi
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_community.document_loaders import PyPDFLoader

In [3]:
loader = PyPDFLoader("NCERT-Class-12-Biology.pdf")
docs = loader.load()
docs

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'NCERT-Class-12-Biology.pdf', 'total_pages': 311, 'page': 0, 'page_label': '1'}, page_content="• • \nllim.tll'l II"),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'NCERT-Class-12-Biology.pdf', 'total_pages': 311, 'page': 1, 'page_label': '2'}, page_content='FoREWORD \nPREFACE \nUNIT VI \nCONTENTS \nREPRODUCTION \nChapter 1 \nChapter 2 \nChapter 3 \nChapter 4 \nUNIT VII \n: Reproduction in Organisms \n: Sexual Reproduction in F1owering Plants \n: Human Reproduction \n: Reproductive Health \nGENETICS AND EvOLUTION \nChapter 5 \nChapter 6 \nChapter 7 \nUNIT VIII \n: Principles of Inheritance and Variation \n: Molecular Basis of Inheritance \n: Evolution \nBIOWGY IN HUMAN WELFARE \nChapter 8 : Human Health and Disease \nChapter 9 : Strategies for Enhancement in \nFood Production \nChapter 10 : Microbes in Human Welfare \nUNIT IX \niii \nvii \n1-66 \n3 \n1

In [4]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter
text_splitter= RecursiveCharacterTextSplitter(
                                chunk_size=1000,
                                chunk_overlap=200)

document = text_splitter.split_documents(docs)

In [5]:
from langchain_ollama import OllamaEmbeddings # convert text -> to vector(numbers)
from langchain_community.vectorstores import Chroma, FAISS 

def cleantext(text):
    return text.encode("utf-8","ignore").decode("utf-8","ignore")

# cleaned= [cleantext(i.page_content) for i in document]
cleaned_docs=[]
for d in document:
    d.page_content= cleantext(d.page_content)
    cleaned_docs.append(d)

In [6]:
cleaned_docs

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'NCERT-Class-12-Biology.pdf', 'total_pages': 311, 'page': 0, 'page_label': '1'}, page_content="• • \nllim.tll'l II"),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'NCERT-Class-12-Biology.pdf', 'total_pages': 311, 'page': 1, 'page_label': '2'}, page_content='FoREWORD \nPREFACE \nUNIT VI \nCONTENTS \nREPRODUCTION \nChapter 1 \nChapter 2 \nChapter 3 \nChapter 4 \nUNIT VII \n: Reproduction in Organisms \n: Sexual Reproduction in F1owering Plants \n: Human Reproduction \n: Reproductive Health \nGENETICS AND EvOLUTION \nChapter 5 \nChapter 6 \nChapter 7 \nUNIT VIII \n: Principles of Inheritance and Variation \n: Molecular Basis of Inheritance \n: Evolution \nBIOWGY IN HUMAN WELFARE \nChapter 8 : Human Health and Disease \nChapter 9 : Strategies for Enhancement in \nFood Production \nChapter 10 : Microbes in Human Welfare \nUNIT IX \niii \nvii \n1-66 \n3 \n1

In [ ]:
emb= OllamaEmbeddings(model="qwen3-embedding:0.6b") 
db= FAISS.from_documents(cleaned_docs, emb)



# emb= OllamaEmbeddings(model="nomic-embed-text") 
# db= FAISS.from_documents(cleaned_docs, emb)

In [8]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
You are an expert research assistant.

Use ONLY the provided context to answer the question.
- Do not use prior knowledge
- If answer is missing, say: "Not found in the document."

Be concise and precise.

Context:
{context}

Question:
{question}

Answer:
"""
)




In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:1b",  #"llama2"
    temperature=0.2
)


In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser 
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt 
    | llm
    | StrOutputParser()
)

In [15]:
query = "What are Molecular Diagnosis?"

response = rag_chain.invoke(query)
print(response)

 Not found in the document.
